# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlahimani/Flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why
*Which method from the toolkit, and why it fits your lane.*

Method choice: Logistic Regression

Why it fits: My question is a yes/no question (does a page have high CTR or not). The training-honest-models guide says yes/no questions with an observed label should start with Logistic Regression, then try Random Forest if needed.

Logistic Regression fits my lane (Ranking Signal Analysis) well because it's simple and easy to read ,it tells you which features push the prediction up or down.My goal is figuring out which signals matter for CTR, a simple, readable model is more useful.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design
*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: Grouped by client_id

I split my data by client_id instead of splitting randomly. If I split randomly, the same client could end up in both my training data and my testing data. If that happens, my model won't really be tested properly,it will just repeat the answer from data it has already seen for that client, instead of actually figuring out the pattern for new client. So I grouped the split by client_id to make sure no client shows up in both sides.

I did not need a time-based split because my CTR label is just one number based on the last 90 days, not something comparing past vs future.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/peddikotlahimani/Flyrank-internship/main/data/raw/content_refresh_anonymized.csv")

print("Loaded! Number of rows:", len(df))
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))


train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

overlap = train_clients & test_clients
print("Clients in both train and test:", len(overlap))

Loaded! Number of rows: 30000
Training rows: 23837
Testing rows: 6163
Clients in both train and test: 0


## 3. Train + compare vs my baseline
*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I trained a computer model to guess if a page has good CTR, using only word count, impressions, and how recently it was updated (not CTR itself, since that would be cheating). I compared its top 50 guesses to my baseline's top 50 guesses.

Result:
Base rate: 0.13
Baseline precision@50: 1.00
Model precision@50: 0.14

My baseline got all 50 right, but that's because my baseline rule already looks at the real CTR number to make its score, so it's basically looking at answer key. My model never got to see CTR at all, so it had a much harder job and only did a little better than guessing randomly. This isn't really a fair fight, because baseline had an unfair advantage. This is a real problem with how I built the baseline, not proof that the model is bad.

I noticed this comparison wasn't fair, since my baseline used the real CTR value to build its score, but my model wasn't allowed to see CTR at all. So I built a second, fair baseline using only the same three signals the model gets (word count, impressions, freshness) and compared again properly.

Result: Base rate: 0.13 Fair baseline precision@50: 0.18 Model precision@50: 0.14

My simple rule did a little better than my model. Both were only a bit better than just guessing randomly, so neither one is really "good" at predicting CTR. This tells me that word count, impressions, and how recently a page was updated just aren't strong enough clues to guess CTR well. This actually matches what I found earlier — longer content didn't even help CTR (shorter pages did better), so it makes sense that these same weak clues don't help the model much either. My simple rule held up fine here, which shows that a fancier model doesn't automatically mean better results, especially when the clues you're feeding it aren't very strong to begin with.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#recreate the same label as baseline
average_ctr = df["ctr"].mean()
df["high_ctr"] = (df["ctr"] >= average_ctr).astype(int)

#recreate same baseline rule
def figure_out_reason(row):
    if pd.isna(row["word_count"]):
        return "missing_content_data"
    elif row["impressions_90d"] < 100:
        return "not_enough_visibility"
    elif row["ctr"] >= average_ctr:
        return "good_ctr"
    else:
        return "low_ctr_but_visible"

df["reason_code"] = df.apply(figure_out_reason, axis=1)

score_map = {"good_ctr": 3, "low_ctr_but_visible": 2, "not_enough_visibility": 1, "missing_content_data": 0}
df["baseline_score"] = df["reason_code"].map(score_map)

print("Done! Labels and baseline score added.")

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

Done! Labels and baseline score added.
Train rows: 23837
Test rows: 6163


In [4]:
# This cell is for CODE (numbers, a query, a check).
# these are the safe features -- no trend_direction, trend_pct, provider_used, model_used
features = ["word_count", "impressions_90d", "days_since_last_update"]

# fill missing word_count so the model doesn't crash on blanks
train_df = train_df.copy()
test_df = test_df.copy()
train_df["word_count"] = train_df["word_count"].fillna(train_df["word_count"].median())
test_df["word_count"] = test_df["word_count"].fillna(train_df["word_count"].median())

X_train = train_df[features]
y_train = train_df["high_ctr"]

X_test = test_df[features]
y_test = test_df["high_ctr"]

print("Features used:", features)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

print("Model trained!")
# get the model's confidence score for each test page (how likely it is to be high CTR)
test_df["model_score"] = model.predict_proba(X_test)[:, 1]

print("Done! Model scores added.")
test_df[["content_id", "model_score", "baseline_score", "high_ctr"]].head(10)

Features used: ['word_count', 'impressions_90d', 'days_since_last_update']
Model trained!
Done! Model scores added.


,content_id,model_score,baseline_score,high_ctr
0,content_304f48230142,0.134822,3,1
1,content_a1fb4e703a9e,0.152320,2,0
5,content_d4084a4bc775,0.136313,2,0
13,content_a5a2fbc76336,0.146411,2,0
19,content_af865035b328,0.136898,1,1
20,content_0d748c484ab1,0.138828,3,1
22,content_3fb46bec4413,0.135687,2,0
23,content_2da6ae9d0882,0.134342,0,0
25,content_033ae3e7aecf,0.135837,1,0
26,content_72c5c2d73e5a,0.139110,2,0


In [5]:
# This cell is for CODE (numbers, a query, a check).
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

k = 50
base_rate = test_df["high_ctr"].mean()

baseline_p_at_k = precision_at_k(test_df["baseline_score"], test_df["high_ctr"], k)
model_p_at_k = precision_at_k(test_df["model_score"], test_df["high_ctr"], k)

print(f"Base rate: {base_rate:.2f}")
print(f"Baseline precision@{k}: {baseline_p_at_k:.2f}")
print(f"Model precision@{k}: {model_p_at_k:.2f}")

Base rate: 0.13
Baseline precision@50: 1.00
Model precision@50: 0.14


In [6]:
# This cell is for CODE (numbers, a query, a check).

median_wc = df["word_count"].median()
median_days = df["days_since_last_update"].median()

def fair_baseline(row):
    if pd.isna(row["word_count"]):
        return 0
    score = 0
    if row["word_count"] >= median_wc:
        score += 1
    if row["impressions_90d"] >= 100:
        score += 1
    if row["days_since_last_update"] <= median_days:
        score += 1
    return score

df["fair_baseline_score"] = df.apply(fair_baseline, axis=1)

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]
test_df["model_score"] = model.predict_proba(X_test)[:, 1]

print("Fair baseline score added.")

Fair baseline score added.


/tmp/ipykernel_1295/3942527446.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["model_score"] = model.predict_proba(X_test)[:, 1]


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.